# I2V Chaining Test: Last-Frame Handoff

**Goal:** Test whether chaining Image-to-Video generations via the last frame produces a natural, continuous narrative flow between clips.

**Method:**
1. Generate Clip 1 from `legoman.png` + a text prompt
2. Extract the last frame of Clip 1
3. Feed that last frame + a new prompt into a second I2V generation (Clip 2)
4. Compare both clips to see if the transition feels coherent

## Setup

In [ ]:
import os
import time
import base64
import requests
from http import HTTPStatus
from pathlib import Path

import cv2
from dotenv import load_dotenv
import dashscope
from dashscope import VideoSynthesis

load_dotenv("../.env")

api_key = os.getenv("DASHSCOPE_API_KEY")
dashscope.base_http_api_url = "https://dashscope-intl.aliyuncs.com/api/v1"

# Output directory for this notebook
OUTPUT_DIR = Path(".")


## Helper Functions

In [ ]:
def wait_for_task(task_id: str, poll_interval: int = 15) -> dict | None:
    """Poll DashScope until the video task succeeds or fails."""
    print(f"Polling task {task_id} every {poll_interval}s...")
    while True:
        status = VideoSynthesis.fetch(task_id, api_key=api_key)
        if status.status_code != HTTPStatus.OK:
            print(f"Poll request failed: {status.code} — {status.message}")
            return None
        task_status = status.output.task_status
        print(f"  Status: {task_status}")
        if task_status == "SUCCEEDED":
            return status
        elif task_status in ("FAILED", "CANCELED", "UNKNOWN"):
            print(f"  code:    {status.output.get('code', 'N/A')}")
            print(f"  message: {status.output.get('message', 'N/A')}")
            return None
        time.sleep(poll_interval)


def download_video(url: str, filename: str) -> Path:
    """Download a video from URL and return the local path."""
    path = OUTPUT_DIR / filename
    print(f"Downloading video to {path}...")
    response = requests.get(url, stream=True)
    with open(path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved: {path}")
    return path


def encode_image_to_data_url(image_path: str) -> str:
    """Read a local image file and return a Base64 data URL for the DashScope API."""
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    # Determine MIME type from extension
    ext = Path(image_path).suffix.lower()
    mime = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
            ".bmp": "image/bmp", ".webp": "image/webp"}.get(ext, "image/png")
    return f"data:{mime};base64,{b64}"


def extract_last_frame(video_path: str, output_path: str) -> str:
    """Extract the last frame of a video and save it as a PNG image."""
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video has {total_frames} frames")

    # Seek to the last frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames - 1)
    ret, frame = cap.read()
    cap.release()

    if not ret:
        raise RuntimeError(f"Failed to read last frame from {video_path}")

    cv2.imwrite(str(output_path), frame)
    print(f"Extracted last frame -> {output_path}")
    return str(output_path)


def generate_i2v(image_path: str, prompt: str, model: str, filename: str,
                 resolution: str = "480P", duration: int = 5) -> Path | None:
    """Run an I2V generation and return the path to the downloaded video."""
    img_url = encode_image_to_data_url(image_path)

    rsp = VideoSynthesis.async_call(
        api_key=api_key,
        model=model,
        img_url=img_url,
        prompt=prompt,
        resolution=resolution,
        duration=duration,
        prompt_extend=False,  # Keep prompts exact for controlled chaining
        watermark=False,
    )

    if rsp.status_code != HTTPStatus.OK:
        print(f"Task creation failed: {rsp.code} — {rsp.message}")
        return None

    task_id = rsp.output.task_id
    print(f"Task created: {task_id}")

    result = wait_for_task(task_id)
    if result:
        return download_video(result.output.video_url, filename)
    return None

## Configuration

Choose the model and parameters for the chaining test. Using `wan2.2-i2v-flash` for fast, cheap iterations. The two prompts should describe a continuous narrative where Clip 2 naturally follows Clip 1.

In [6]:
# --- Chaining config ---

MODEL = "wan2.2-i2v-flash"   # Fast and cheap, good for iteration
RESOLUTION = "480P"           # Keep low for faster testing
DURATION = 5                  # Fixed 5s for wan2.2

# Source image
SOURCE_IMAGE = "../legoman.png"

# Clip 1: Initial animation from the source image
PROMPT_CLIP_1 = (
    "The lego figure slowly turns its head to the right and raises one arm, "
    "as if waving hello. Smooth, gentle motion."
)

# Clip 2: Continuation from the last frame of Clip 1
PROMPT_CLIP_2 = (
    "The lego figure lowers its arm and begins walking forward step by step. "
    "The camera follows along smoothly."
)

# Output filenames
CLIP_1_FILE = "chain_clip_1.mp4"
CLIP_2_FILE = "chain_clip_2.mp4"
LAST_FRAME_FILE = "chain_clip_1_last_frame.png"

## Step 1: Generate Clip 1 from the source image

In [ ]:
clip_1_path = generate_i2v(
    image_path=SOURCE_IMAGE,
    prompt=PROMPT_CLIP_1,
    model=MODEL,
    filename=CLIP_1_FILE,
    resolution=RESOLUTION,
    duration=DURATION,
)
print(f"\nClip 1: {clip_1_path}")

## Step 2: Extract the last frame of Clip 1

This frame becomes the conditioning image for Clip 2, creating a visual bridge between the two generations.

In [ ]:
last_frame_path = extract_last_frame(clip_1_path, OUTPUT_DIR / LAST_FRAME_FILE)

# Display the extracted last frame
from IPython.display import display, Image as IPImage
display(IPImage(filename=last_frame_path))

## Step 3: Generate Clip 2 from the last frame

Using the extracted last frame as the new conditioning image with a continuation prompt.

In [ ]:
clip_2_path = generate_i2v(
    image_path=last_frame_path,
    prompt=PROMPT_CLIP_2,
    model=MODEL,
    filename=CLIP_2_FILE,
    resolution=RESOLUTION,
    duration=DURATION,
)
print(f"\nClip 2: {clip_2_path}")

## Step 4: Review both clips

Display both videos side by side for visual inspection. Look for:
- **Visual continuity**: Does the first frame of Clip 2 match the last frame of Clip 1?
- **Motion continuity**: Does the movement feel like a natural continuation?
- **Style consistency**: Are lighting, colors, and rendering style preserved?

In [ ]:
from IPython.display import display, Video, HTML

print("=== Clip 1 ===")
display(Video(str(clip_1_path), embed=True, mimetype="video/mp4", width=480))

print("\n=== Clip 2 (chained from last frame) ===")
display(Video(str(clip_2_path), embed=True, mimetype="video/mp4", width=480))

## Step 5: Side-by-side first/last frame comparison

Compare the transition point: last frame of Clip 1 vs first frame of Clip 2.

In [ ]:
import matplotlib.pyplot as plt

# Last frame of Clip 1 (already extracted)
last_frame_clip1 = cv2.imread(str(OUTPUT_DIR / LAST_FRAME_FILE))
last_frame_clip1 = cv2.cvtColor(last_frame_clip1, cv2.COLOR_BGR2RGB)

# First frame of Clip 2
cap = cv2.VideoCapture(str(clip_2_path))
ret, first_frame_clip2 = cap.read()
cap.release()
first_frame_clip2 = cv2.cvtColor(first_frame_clip2, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(last_frame_clip1)
axes[0].set_title("Last frame of Clip 1\n(input to Clip 2)")
axes[0].axis("off")

axes[1].imshow(first_frame_clip2)
axes[1].set_title("First frame of Clip 2\n(generated)")
axes[1].axis("off")

plt.suptitle("Transition Point Comparison", fontsize=14)
plt.tight_layout()
plt.show()

## Step 6: Merge clips into a single video

Concatenate Clip 1 and Clip 2 into one continuous video to judge the transition in real time.

In [9]:
MERGED_FILE = "chain_merged.avi"
clip_1_path = OUTPUT_DIR / CLIP_1_FILE
clip_2_path = OUTPUT_DIR / CLIP_2_FILE

cap1 = cv2.VideoCapture(str(clip_1_path))
fps = cap1.get(cv2.CAP_PROP_FPS)
width = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter(str(OUTPUT_DIR / MERGED_FILE), fourcc, fps, (width, height))

# Write all frames from Clip 1
while True:
    ret, frame = cap1.read()
    if not ret:
        break
    out.write(frame)
cap1.release()

# Write all frames from Clip 2
cap2 = cv2.VideoCapture(str(clip_2_path))
while True:
    ret, frame = cap2.read()
    if not ret:
        break
    if (frame.shape[1], frame.shape[0]) != (width, height):
        frame = cv2.resize(frame, (width, height))
    out.write(frame)
cap2.release()
out.release()

print(f"Merged video saved: {MERGED_FILE} ({fps:.1f} fps, {width}x{height})")
print("Note: .avi file — open it in a local video player (VLC, etc.) to review.")

Merged video saved: chain_merged.avi (30.0 fps, 624x624)
Note: .avi file — open it in a local video player (VLC, etc.) to review.
